# TUGAS MANDIRI

### Konteks / Skenario

Tim data engineering platform e-commerce meminta anda membangun pipeline ETL produksi pertama yang menggabungkan tiga sumber data sekaligus: transaksi (CSV), data produk (JSON), dan data ulasan pelanggan (CSV terpisah) — merepresentasikan kondisi nyata di mana data tersebar di berbagai sistem dengan format berbeda-beda.

### SparkSession

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, count, row_number
from pyspark.sql.window import Window
import pandas as pd

# Nyalakan SparkSession
spark = SparkSession.builder \
    .appName("Tugas6_Praktikum_BigData") \
    .master("local[*]") \
    .getOrCreate()

# Matikan log warning supaya output bersih
spark.sparkContext.setLogLevel("ERROR")

26/09/29 08:18:09 WARN Utils: Your hostname, rindani-Latitude-3410 resolves to a loopback address: 127.0.1.1; using 192.168.1.11 instead (on interface wlp0s20f3)
26/09/29 08:18:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/29 08:18:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### Menyiapkan Dataset

In [1]:
import numpy as np
import pandas as pd
import json

np.random.seed(33)

# Sumber 1: Transaksi (CSV) — 5000 baris
n_trx = 5000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
df_trx_tugas6 = pd.DataFrame({
    "order_id": [f"TX{i}" for i in range(n_trx)],
    "product_id": np.random.randint(1, 31, size=n_trx),
    "unit_terjual": np.random.randint(1, 8, size=n_trx),
    "tanggal": np.random.choice(pd.date_range("2026-10-01","2026-10-31"), size=n_trx).astype(str),
})
df_trx_tugas6.to_csv("tugas6_transaksi.csv", index=False)

# Sumber 2: Data produk (JSON Lines) — 30 produk
produk = [
    {"product_id": i, "nama_produk": f"Produk-{i}", "kategori": np.random.choice(kategori_list),
     "harga": int(np.random.choice([25000,50000,75000,100000,150000,250000]))}
    for i in range(1, 31)
]
with open("tugas6_produk.json", "w") as f:
    for p in produk:
        f.write(json.dumps(p) + "\n")

# Sumber 3: Data ulasan (CSV) — tidak semua transaksi memiliki ulasan (realistis)
n_review = 3500
df_review = pd.DataFrame({
    "order_id": np.random.choice(df_trx_tugas6["order_id"], size=n_review, replace=False),
    "rating": np.random.randint(1, 6, size=n_review),
})
df_review.to_csv("tugas6_ulasan.csv", index=False)

print(f"Tiga sumber data berhasil dibuat:")
print(f"- tugas6_transaksi.csv : {len(df_trx_tugas6)} baris")
print(f"- tugas6_produk.json   : {len(produk)} baris")
print(f"- tugas6_ulasan.csv    : {len(df_review)} baris (tidak seluruh transaksi memiliki ulasan)")

Tiga sumber data berhasil dibuat:
- tugas6_transaksi.csv : 5000 baris
- tugas6_produk.json   : 30 baris
- tugas6_ulasan.csv    : 3500 baris (tidak seluruh transaksi memiliki ulasan)


### A. EXTRACT 

Baca ketiga sumber data (tugas6_transaksi.csv, tugas6_produk.json, tugas6_ulasan.csv) menjadi tiga Spark DataFrame terpisah. Tampilkan jumlah baris dan printSchema() masing-masing.



In [4]:
# Membaca data transaksi
df_transaksi = spark.read.csv("tugas6_transaksi.csv", header=True, inferSchema=True)

# Membaca data produk (format JSON Lines)
df_produk = spark.read.json("tugas6_produk.json")

# Membaca data ulasan
df_ulasan = spark.read.csv("tugas6_ulasan.csv", header=True, inferSchema=True)

# Menampilkan jumlah baris dan skema masing-masing
print(f"Jumlah baris Transaksi: {df_transaksi.count()}")
df_transaksi.printSchema()

print(f"Jumlah baris Produk: {df_produk.count()}")
df_produk.printSchema()

print(f"Jumlah baris Ulasan: {df_ulasan.count()}")
df_ulasan.printSchema()

Jumlah baris Transaksi: 5000
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

Jumlah baris Produk: 30
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)

Jumlah baris Ulasan: 3500
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



### B. TRANSFORM — Penggabungan 

Gabungkan ketiga DataFrame menjadi satu (order_id sebagai kunci ke ulasan, product_id sebagai kunci ke produk). Gunakan salah satu join yang tepat dari transaksi ke ulasan (karena tidak semua transaksi memiliki ulasan) dan salah satu join yang tepat dari transaksi ke produk (karena setiap transaksi pasti memiliki produk yang valid). Tambahkan kolom total_pendapatan (unit_terjual x harga).

In [5]:
# 1. Join transaksi dengan produk (Inner Join)
df_joined = df_transaksi.join(df_produk, on="product_id", how="inner")

# 2. Join dengan ulasan (Left Join agar transaksi tanpa ulasan tetap ada)
df_joined = df_joined.join(df_ulasan, on="order_id", how="left")

# 3. Menambahkan kolom total_pendapatan
df_joined = df_joined.withColumn("total_pendapatan", col("unit_terjual") * col("harga"))

df_joined.show(5)

+--------+----------+------------+-------------------+------+------------+-----------+------+----------------+
|order_id|product_id|unit_terjual|            tanggal| harga|    kategori|nama_produk|rating|total_pendapatan|
+--------+----------+------------+-------------------+------+------------+-----------+------+----------------+
|     TX0|        21|           2|2026-10-10 00:00:00| 25000|  Elektronik|  Produk-21|     1|           50000|
|     TX1|         8|           6|2026-10-25 00:00:00|250000|     Fashion|   Produk-8|  NULL|         1500000|
|     TX2|        25|           4|2026-10-13 00:00:00| 25000|  Elektronik|  Produk-25|     2|          100000|
|     TX3|         3|           4|2026-10-18 00:00:00| 75000|     Fashion|   Produk-3|     5|          300000|
|     TX4|        19|           6|2026-10-07 00:00:00| 75000|Rumah Tangga|  Produk-19|  NULL|          450000|
+--------+----------+------------+-------------------+------+------------+-----------+------+----------------+
o

### C. TRANSFORM — Penanganan Data Kosong & Pengayaan 

Transaksi tanpa ulasan akan memiliki rating bernilai kosong (null) setelah salah satu join yang tepat — isi nilai kosong tersebut dengan angka 0 menggunakan salah satu function, sertakan alasan singkat mengapa 0 (bukan nilai lain) masuk akal untuk kasus "belum ada ulasan".
 Tambahkan kolom ada_ulasan bernilai True/False (tidak boleh diisi manual satu satu), **sebelum** langkah na.fill()` di atas).



In [7]:
from pyspark.sql.functions import col, when

# 1. Menambahkan kolom ada_ulasan SEBELUM mengisi null
df_transformed = df_joined.withColumn(
    "ada_ulasan",
    when(col("rating").isNotNull(), True).otherwise(False)
)

# 2. Mengisi nilai null pada rating dengan angka 0
df_transformed = df_transformed.na.fill({"rating": 0})

df_transformed.select("order_id", "nama_produk", "rating", "ada_ulasan", "total_pendapatan").show(5)

+--------+-----------+------+----------+----------------+
|order_id|nama_produk|rating|ada_ulasan|total_pendapatan|
+--------+-----------+------+----------+----------------+
|     TX0|  Produk-21|     1|      true|           50000|
|     TX1|   Produk-8|     0|     false|         1500000|
|     TX2|  Produk-25|     2|      true|          100000|
|     TX3|   Produk-3|     5|      true|          300000|
|     TX4|  Produk-19|     0|     false|          450000|
+--------+-----------+------+----------+----------------+
only showing top 5 rows



### D. LOAD 

Simpan hasil akhir ke HDFS dalam format Parquet, dipartisi berdasarkan kategori, ke path /user/[username]/tugas6/hasil_etl. Verifikasi dengan hdfs dfs -ls -R, lalu baca kembali dan tampilkan count()-nya sebagai bukti data tersimpan utuh.

In [8]:
# Path tujuan di HDFS (Ganti 'mahasiswa' dengan username HDFS Anda jika berbeda)
hdfs_path = "hdfs://localhost:9000/user/mahasiswa/tugas6/hasil_etl"

# Membuat direktori induk di HDFS
!hdfs dfs -mkdir -p /user/mahasiswa/tugas6

# Menyimpan ke HDFS dengan format Parquet & terpartisi
df_transformed.write.mode("overwrite").partitionBy("kategori").parquet(hdfs_path)
print("Data berhasil disimpan ke HDFS.")

# Verifikasi struktur folder
!hdfs dfs -ls -R /user/mahasiswa/tugas6/hasil_etl

# Baca kembali dan tampilkan jumlah baris sebagai verifikasi
df_verifikasi = spark.read.parquet(hdfs_path)
print(f"Verifikasi jumlah baris data yang dibaca dari HDFS: {df_verifikasi.count()}")

[Stage 22:>                                                         (0 + 1) / 1]

Data berhasil disimpan ke HDFS.


-rw-r--r--   3 rindani supergroup          0 2026-09-29 08:20 /user/mahasiswa/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - rindani supergroup          0 2026-09-29 08:20 /user/mahasiswa/tugas6/hasil_etl/kategori=Elektronik
-rw-r--r--   3 rindani supergroup      16364 2026-09-29 08:20 /user/mahasiswa/tugas6/hasil_etl/kategori=Elektronik/part-00000-104237a5-05e4-47ac-aff9-353ad71579ed.c000.snappy.parquet
drwxr-xr-x   - rindani supergroup          0 2026-09-29 08:20 /user/mahasiswa/tugas6/hasil_etl/kategori=Fashion
-rw-r--r--   3 rindani supergroup      11134 2026-09-29 08:20 /user/mahasiswa/tugas6/hasil_etl/kategori=Fashion/part-00000-104237a5-05e4-47ac-aff9-353ad71579ed.c000.snappy.parquet
drwxr-xr-x   - rindani supergroup          0 2026-09-29 08:20 /user/mahasiswa/tugas6/hasil_etl/kategori=Kesehatan
-rw-r--r--   3 rindani supergroup      10393 2026-09-29 08:20 /user/mahasiswa/tugas6/hasil_etl/kategori=Kesehatan/part-00000-104237a5-05e4-47ac-aff9-353ad71579ed.c000.snappy.parquet
drwxr-xr-x

### E. Insight Akhir 

Dari data hasil ETL, tampilkan (menggunakan DataFrame API atau Spark SQL, bebas memilih): kategori produk mana yang memiliki persentase transaksi dengan ulasan (ada_ulasan = True) paling rendah? Tulis 2-3 kalimat interpretasi bisnis pada markdown cell: mengapa hal ini mungkin penting diketahui oleh tim marketing?

In [11]:
from pyspark.sql.functions import col, when, count, round as spark_round

# Analisis Menggunakan Spark DataFrame API
df_insight = df_transformed.groupBy("kategori").agg(
    count("*").alias("total_transaksi"),
    count(when(col("ada_ulasan") == True, 1)).alias("transaksi_berulasan")
).withColumn(
    "persentase_ulasan",
    spark_round((col("transaksi_berulasan") / col("total_transaksi")) * 100, 2)
).orderBy("persentase_ulasan")

df_insight.show()

+------------+---------------+-------------------+-----------------+
|    kategori|total_transaksi|transaksi_berulasan|persentase_ulasan|
+------------+---------------+-------------------+-----------------+
|     Makanan|            536|                369|            68.84|
|   Kesehatan|            961|                662|            68.89|
|     Fashion|           1035|                726|            70.14|
|Rumah Tangga|            815|                575|            70.55|
|  Elektronik|           1653|               1168|            70.66|
+------------+---------------+-------------------+-----------------+



### Mengapa Hal Ini Penting bagi Tim Marketing?

Minimnya partisipasi ulasan pada kategori produk ini menunjukkan adanya gap interaksi antara platform dan pelanggan setelah proses jual-beli selesai.

Tim marketing dapat menjadikan temuan ini sebagai dasar untuk mengevaluasi pengalaman pascabeli (post-purchase experience). Dengan menghadirkan program dorongan seperti insentif ulasan, platform tidak hanya memperoleh feedback untuk perbaikan produk, tetapi juga membangun social proof yang lebih kuat untuk memikat pembeli baru di masa mendatang.

### Menutup SparkSession

In [12]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
